# Median Income
This notebook gives a very basic example of using the `miximaps`
library to load a census detail table and display a choropleth
of that data for NYC Census tracts.

In orde to run this Notebook, you will have to
1. Get your own [Census API key](https://api.census.gov/data/key_signup.html)
2. Add it to Colab or a local .env file if you are not in Colab.


## Add a key in Colab
1. Click on the "key" icon on the left
2. CLick **Add new secret**
3. For **Name** enter: `CENSUS_API_KEY`
4. For **Value** enter the key (random number) you received from the US Census in your email.

# Install libraries
"libraries" are bundles python code that add functionality to the core language.
Some packages (e.g. `math`, `random`) are part of the language; others
are written by other developers to add specific functionality.

Here are a few of the libraries we import:
- `pandas` a key data science library for working with large sets of data
- `geopandas` enables `pandas` to work with geospatial data and to produce static and dynamic maps
- `plotly` a library for making interactive graphs and charts
- `census` makes it easier for us to request data directly from the US Census

We also install and import our own library called **miximaps**. This package contains useful functions for working with geospatial data, especially for the Census and NYC. Members of our map club contribute to this library and can help shape it.

In [1]:
# install the miximaps package for our club
# this will also install some other useful package
# that are in Colab by default
!pip install miximaps -qq


In [2]:
from miximaps import nyc,ui

import pandas as pd
from census import Census


# Initialize the Census
Here we load our secret API key. This code is written
so that it can work from your local computer or
from the online Colab IDE without having to be udpated.


In [3]:
api_key = ""
try:
    from google.colab import userdata
    userdata.get('CENSUS_API_KEY')
except ImportError:
    import os
    api_key = os.environ["CENSUS_API_KEY"]

year = 2023
c = Census(api_key, year=year)

# Table B19013: Median income
- use mixi maps to load median household income for our year (2023) for
  all of the census tracts in NYC 5 boroughs and the "inner" counties
- rename the columns to make them shorter
- drop all of the rows that don't have good data



## Load and prep the data

In [4]:
# create table var for median income
table = "B19013"

# load the data
df = nyc.get_tracts(c, table, year=year,region="inner")

# rename the median icnome column
df.rename(columns={"median_household_income_in_the_past_12_months_in_2023_inflation_adjusted_dollars":"median_income"}, inplace=True)

# show the column names
display(df.columns)

# drop empty tracts
df = df[df.median_income > 0]

# show a "sample" of 10 rows of data
df.sample(10)

Index(['geographic_area_name', 'geography', 'median_income', 'state', 'county',
       'tract', 'statefp', 'countyfp', 'geometry', 'borough'],
      dtype='object')

,geographic_area_name,geography,median_income,state,county,tract,statefp,countyfp,geometry,borough
2989,Census Tract 423.01; Bergen County; New Jersey,1400000US34003042301,120208.0,NJ,Bergen County,042301,34,003,"MULTIPOLYGON (((-74.0971 40.96588, -74.09709 4...",-
3074,Census Tract 25; Essex County; New Jersey,1400000US34013002500,78147.0,NJ,Essex County,002500,34,013,"MULTIPOLYGON (((-74.22576 40.74077, -74.22277 ...",-
1061,Census Tract 934; Kings County; New York,1400000US36047093400,83357.0,NY,Kings County,093400,36,047,"MULTIPOLYGON (((-73.92077 40.65255, -73.91996 ...",Brooklyn
2951,Census Tract 304.01; Bergen County; New Jersey,1400000US34003030401,89275.0,NJ,Bergen County,030401,34,003,"MULTIPOLYGON (((-74.09141 40.86466, -74.09088 ...",-
927,Census Tract 652; Kings County; New York,1400000US36047065200,92969.0,NY,Kings County,065200,36,047,"MULTIPOLYGON (((-73.9332 40.61697, -73.93249 4...",Brooklyn
1416,Census Tract 223.02; New York County; New York,1400000US36061022302,39004.0,NY,New York County,022302,36,061,"POLYGON ((-73.95864 40.82158, -73.95853 40.821...",Manhattan
1170,Census Tract 7; New York County; New York,1400000US36061000700,197058.0,NY,New York County,000700,36,061,"MULTIPOLYGON (((-74.01195 40.70745, -74.0117 4...",Manhattan
1669,Census Tract 94; Queens County; New York,1400000US36081009400,72130.0,NY,Queens County,009400,36,081,"MULTIPOLYGON (((-73.83953 40.68126, -73.83865 ...",Queens
1697,Census Tract 123.01; Queens County; New York,1400000US36081012301,109250.0,NY,Queens County,012301,36,081,"MULTIPOLYGON (((-73.90596 40.77619, -73.90502 ...",Queens
2551,Census Tract 148.12; Westchester County; New York,1400000US36119014812,183500.0,NY,Westchester County,014812,36,119,"MULTIPOLYGON (((-73.8561 41.30046, -73.8552 41...",-


## Make a map
This "choropleth" uses shades of Purple to show differenes in median income.
Darker purple tracts have a higher median income. Note that some tracts are empty because we did not have valid data returned for thos tracts.

In [6]:
# create a new column that formats median income as a whole dollar string
df["Median Income"] = df["median_income"].apply(lambda x: f"${x:,.0f}")

# list the columns we want to show in the popup,
# in the order we want them to appear
cols =["geographic_area_name", "Median Income", "county", "state"]

# create a new column with the pop info
df["popup"] = df.apply(ui.popup(cols), axis=1)

# get a basemap using the (default) cartodb tiles
m = ui.base_map(df)

# create the map
m = df.explore(m=m, column="median_income", tooltip="Median Income",
    popup="popup", cmap="Purples", popup_kwds=dict(labels=False),
    style_kwds=dict(fillOpacity=1, opacity=1))
m.save("median-inc.html")
